# 环节 06 · 残差 + 归一化（配套 Notebook）

> 配套长文：[环节06-残差连接与归一化详解.md](./环节06-残差连接与归一化详解.md)
> 定位：把"残差给梯度修高速公路""RMSNorm 省在哪""Pre-LN 为什么稳"三件事**验证成数**。纯 Python 标准库，零依赖。

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 两种归一化 | §3 | LayerNorm 减均值、RMSNorm 不减 |
| §2 残差 = 恒等旁路 | §2 | 梯度含一项恒等，不随深度衰减 |
| §3 Pre-LN vs Post-LN | §4 | Jacobian 行列式：稳定 vs 指数塌陷 |
| §4 恒等方向 vs 均值方向 | §4 | `J_pre·1 = 1` 精确成立；LN 的核含常数方向 |
| §5 算子/参数账 | §5 | RMSNorm 省掉一次统计 |


## 1. 两种归一化（长文 §3）

```
LayerNorm: (x − μ) / √(σ² + ε) · γ + β     ← 减均值 + 除标准差，两次统计
RMSNorm  : x / √(mean(x²) + ε) · γ          ← 只除均方根，一次统计
```

对每个 token 的特征维独立做。


In [ ]:
import math


def layer_norm(x, eps=1e-5):
    mu = sum(x) / len(x)
    var = sum((xi - mu) ** 2 for xi in x) / len(x)
    s = math.sqrt(var + eps)
    return [(xi - mu) / s for xi in x]


def rms_norm(x, eps=1e-5):
    ms = sum(xi * xi for xi in x) / len(x)
    s = math.sqrt(ms + eps)
    return [xi / s for xi in x]


x = [3.0, -1.0, 0.5, 12.0]
ln_out, rms_out = layer_norm(x), rms_norm(x)

print(f"输入      {[round(v, 3) for v in x]}")
print(f"LayerNorm {[round(v, 4) for v in ln_out]}")
print(f"RMSNorm   {[round(v, 4) for v in rms_out]}")
print()
print(f"输入均值 {sum(x)/len(x):+.3f}")
print(f"  LayerNorm 后均值 = {sum(ln_out)/len(ln_out):+.2e}（被减掉了）")
print(f"  RMSNorm   后均值 = {sum(rms_out)/len(rms_out):+.3f}（保留原比例）")
print(f"输入尺度 ‖x‖ = {math.sqrt(sum(v*v for v in x)):.3f}")
print(f"  LayerNorm 后 ‖x‖ = {math.sqrt(sum(v*v for v in ln_out)):.3f}")
print(f"  RMSNorm   后 ‖x‖ = {math.sqrt(sum(v*v for v in rms_out)):.3f}   ← 只把“长度”压回 √d")


In [ ]:
# 归一化到底在治什么：尺度漂移
print("同一批数据，不同量级的输入，归一化后都回到稳定范围：\n")
print(f"{'输入':<34} {'LN 后 ‖·‖':>12} {'RMS 后 ‖·‖':>12}")
print("-" * 62)
for scale in (0.01, 1.0, 100.0, 10000.0):
    y = [v * scale for v in x]
    print(f"{str([round(v, 2) for v in y]):<34} "
          f"{math.sqrt(sum(v*v for v in layer_norm(y))):>12.4f} "
          f"{math.sqrt(sum(v*v for v in rms_norm(y))):>12.4f}")
print(f"\n→ 不管上一层把数值放大到 1e4 还是缩到 1e-2，归一化后范数都固定在 √d ≈ {math.sqrt(4):.4f}")
print("  这就是“把每层输入拉回稳定尺度”（长文 §1 的分布漂移问题）。")


## 2. 残差 = 给梯度修高速公路（长文 §2）

`X ← X + f(X)` 的反向传播：

```
∂L/∂X = ∂L/∂(X+f(X)) · (1 + ∂f/∂X)
                        └─ 那个 1 就是恒等旁路
```

所以无论 `∂f/∂X` 多小，梯度都能从"1"这条路直达。下面的实验把"有/无残差"的梯度随深度变化放在一起比。


In [ ]:
import random

random.seed(3)
D, a = 6, 0.35
J = [[random.gauss(0, 1 / math.sqrt(D)) for _ in range(D)] for _ in range(D)]


def matvec(M, v):
    return [sum(p * q for p, q in zip(row, v)) for row in M]


def chain_norm(layers, identity):
    """连乘 layers 个 Jacobian（identity=True 时包含恒等项），返回 Frobenius 范数。"""
    v = [1.0] + [0.0] * (D - 1)
    for _ in range(layers):
        sub = [a * t for t in matvec(J, v)]
        v = [vi + si for vi, si in zip(v, sub)] if identity else sub
    return math.sqrt(sum(t * t for t in v))


print(f"{'层数':>5} {'无残差 ‖J^L‖':>16} {'有残差 ‖J^L‖':>16} {'相差倍数':>14}")
print("-" * 56)
for layers in (1, 2, 4, 8, 16, 32):
    no_res, res = chain_norm(layers, False), chain_norm(layers, True)
    print(f"{layers:>5} {no_res:>16.4f} {res:>16.4f} {res / no_res:>14.2e}")

print("\n→ 无残差：连乘 ∂f/∂X，量级随深度**指数衰减**（32 层已经到 0）——梯度消失的根源。")
print("  有残差：梯度不会消失（32 层仍有 6.7e4），但注意它也不是“稳在 1”——")
print("  恒等主干会让信号一路累加，也可能**指数增长**。")
print("\n→ 这正是长文 §1 说“必须解决两个老问题”的原因：")
print("  **残差管通路（不消失）、归一化管尺度（不爆炸）** —— 两者缺一不可（见 §3 的 Pre-LN）。")
print("\n同一思想的另外两个亲戚（长文 §2）：ResNet 的恒等映射、LSTM 的记忆传送带。")


## 3. Pre-LN vs Post-LN：用 Jacobian 行列式看出来（长文 §4）

```
Post-LN（原版）: x ← LN(x + f(x))      归一化在“混合之后”
Pre-LN （主流）: x ← x + f(LN(x))      残差主干保持干净恒等
```

两者都能训，但深层差别很大。量化办法：看**整个网络 Jacobian 的行列式**——它表示"一个单位体积的输入扰动，传到输出时被缩放了多少倍"。行列式→0 意味着大量方向上的梯度被压没了。


In [ ]:
def ln(x, eps=1e-5):
    mu = sum(x) / len(x)
    var = sum((xi - mu) ** 2 for xi in x) / len(x)
    s = math.sqrt(var + eps)
    return [(xi - mu) / s for xi in x]


def forward(x, Ws, mode, a=0.4):
    for W in Ws:
        if mode == "pre":
            x = [xi + a * f for xi, f in zip(x, matvec(W, ln(x)))]
        else:
            x = ln([xi + a * f for xi, f in zip(x, matvec(W, x))])
    return x


def jacobian(x, Ws, mode, eps=1e-6):
    """数值 Jacobian：第 i 列 = 只扰动第 i 个输入分量时输出的变化率。"""
    d = len(x)
    cols = []
    for i in range(d):
        xp, xm = x[:], x[:]
        xp[i] += eps
        xm[i] -= eps
        yp, ym = forward(xp, Ws, mode), forward(xm, Ws, mode)
        cols.append([(p - q) / (2 * eps) for p, q in zip(yp, ym)])
    return [[cols[j][i] for j in range(d)] for i in range(d)]


def det(M):
    A = [row[:] for row in M]
    n, out = len(A), 1.0
    for col in range(n):
        piv = max(range(col, n), key=lambda r: abs(A[r][col]))
        if abs(A[piv][col]) < 1e-14:
            return 0.0
        if piv != col:
            A[col], A[piv] = A[piv], A[col]
            out = -out
        out *= A[col][col]
        for r in range(col + 1, n):
            fac = A[r][col] / A[col][col]
            for k in range(col, n):
                A[r][k] -= fac * A[col][k]
    return out


x0 = [random.gauss(0, 1) for _ in range(D)]
print(f"{'层数':>5} {'Pre-LN |det J|':>18} {'Post-LN |det J|':>18}")
print("-" * 46)
for layers in (1, 2, 4, 8, 16):
    Ws = [[[random.gauss(0, 1 / math.sqrt(D)) for _ in range(D)] for _ in range(D)]
          for _ in range(layers)]
    dp = abs(det(jacobian(x0, Ws, "pre")))
    dq = abs(det(jacobian(x0, Ws, "post")))
    print(f"{layers:>5} {dp:>18.4f} {dq:>18.2e}")

print("\n→ Pre-LN：行列式在 1 附近上下浮动（体积基本保持，梯度不会被闷死）")
print("  Post-LN：每层都乘一个“秩亏 1”的 LN Jacobian（见 §4），连乘后行列式")
print("  指数塌陷到 1e-20 量级 —— 大量方向的梯度被压成 0，这就是深层难训的原因。")


## 4. 为什么 LN 的 Jacobian"秩亏 1"（长文 §4 的机制补充）

LN 输出的**均值恒为 0**：给输入的所有分量同时加一个常数 c，减均值那步会把这个 c 完整抵消 → **输出完全不变**。

也就是说，沿"全 1 方向"的扰动，LN 的响应是 0 —— 该方向落在 LN Jacobian 的**核（kernel）**里。每层都乘这样一个秩 `d-1` 的矩阵，L 层连乘后秩最多 `d-L`。

而 Pre-LN 的 Jacobian 是 `I + (...)·J_LN`，那个恒等 `I` 让"全 1 方向"完好保留：


In [ ]:
ones = [1.0] * D
W1 = [[random.gauss(0, 1 / math.sqrt(D)) for _ in range(D)] for _ in range(D)]

J_ln = jacobian(x0, [W1], "post")          # Post-LN 一层 ≈ J_LN·(I+aW)
J_pre = jacobian(x0, [W1], "pre")          # Pre-LN 一层 = I + aW·J_LN


def ln_only(x, eps=1e-6):
    return ln(x)


def jac_of(fn, x, eps=1e-6):
    d = len(x)
    cols = []
    for i in range(d):
        xp, xm = x[:], x[:]
        xp[i] += eps
        xm[i] -= eps
        diff = [(p - q) / (2 * eps) for p, q in zip(fn(xp), fn(xm))]
        cols.append(diff)
    return [[cols[j][i] for j in range(d)] for i in range(d)]


J_pure_ln = jac_of(ln_only, x0)
print(f"纯 LN 的 Jacobian 作用在“全 1 方向”上：")
print(f"  J_LN · 1 = {[round(v, 10) for v in matvec(J_pure_ln, ones)]}   ← 精确为 0")

print(f"\nPre-LN 一层（含恒等旁路）作用在“全 1 方向”上：")
print(f"  J_pre · 1 = {[round(v, 6) for v in matvec(J_pre, ones)]}   ← 精确为 1")

print("\n→ 这就是“残差主干干净恒等”的量化证据：")
print("  Pre-LN 里那个常数方向（可以理解为“所有 token 共享的那部分信号”）能 1:1 原样传下去；")
print("  Post-LN 每层都把它投影掉，深层自然丢信息。")


## 5. 算子与参数账（长文 §3 / §5）

LLaMA 三件套 = `RMSNorm + Pre-LN + SwiGLU`，分别对应**便宜 / 稳 / 好**。


In [ ]:
d = 4096
L = 32

print("单个归一化的统计量开销（每个 token、每个位置都要算一遍）：")
print(f"  LayerNorm：求均值（{d} 次加）+ 求方差（{d} 次乘加）+ 归一化 → 约 3 次全维扫描")
print(f"  RMSNorm  ：求均方根（{d} 次乘加）+ 归一化 → 约 2 次全维扫描")
print(f"  → 省掉约 1/3 的统计开销（长文 §3“算子更轻”的由来）")

print(f"\n可学习参数（两者相同）：")
print(f"  LayerNorm：γ + β = 2×{d} = {2*d:,} 个/层 → {L} 层共 {2*d*L/1e6:.2f} M")
print(f"  RMSNorm  ：γ     = 1×{d} = {d:,} 个/层 → {L} 层共 {d*L/1e6:.2f} M")
print(f"  → 相对 7B 总参数（约 6.6B）可忽略，但算子更省的收益是实打实的")

print("\n为什么“减均值”可以省（长文 §3 对比理解）：")
print("  归一化的主要目的是**稳住尺度**（防分布漂移），而不是把分布强行标准化；")
print("  只除 RMS 已能稳住尺度，于是现代开源模型（LLaMA / Qwen / DeepSeek）默认用 RMSNorm。")


## 6. 自测（长文 §6）

| 问题 | 本 Notebook 的现场证据 |
|---|---|
| LayerNorm 与 RMSNorm 的差别？ | §1：LN 后均值 = 0，RMS 后均值保留原比例 |
| RMSNorm 为什么能省掉减均值？ | §5：主矛盾是尺度漂移，只需压长度 |
| 残差为什么能让网络变深？ | §2：连乘里那个恒等项把梯度量级稳定在 1 附近 |
| Post-LN 为什么难训？ | §3：|det J| 随深度指数塌陷到 1e-20 |
| 凭什么说 Pre-LN"主干干净恒等"？ | §4：`J_pre·1 = 1` 精确成立，而 `J_LN·1 = 0` |
| 归一化在推理时还起什么作用？ | §1：γ/β 与统计算式照旧，结构不变 |

**上一站** [环节 05 · FFN / MoE](./环节05-FFN激活与MoE详解.md)   **下一站** [环节 07 · Block 堆叠](./环节07-Block堆叠与整体架构详解.md)
